# 06 · Case Study Starter Pack

**Agentic AI for Actuaries** · IFoA Workshop · 10 July 2026 · Hub: `github.com/rohanyashraj/ifoa-workshop`

> All data in this notebook is **hypothetical** — ABC Insurer is a fictional entity calibrated to plausible Indian market experience, for teaching only.

**Mandatory group case study** · teams of ~5 · due **24 July 2026** · mentor office hours Tue & Thu.

Pick ONE track. Every track ships a **working, governed agent** plus the written governance pack.

| Track | Build | Start from |
|---|---|---|
| 1 · Model-tool agent | Wrap a GLM + XGBoost pipeline as governed tools; agent runs, compares, explains on a fresh dataset | Notebooks 02 + 05 |
| 2 · Document & RAG agent | Documentation agent grounded by RAG over a methodology pack, with logging + reviewer gate | Notebooks 01 + 05 |
| 3 · MCP automation agent | MCP + Claude Desktop automating one actuarial task end-to-end | Notebook 04 + MCP template |

**Deliverables:** (a) runnable notebook/repo · (b) spec + system prompt as governed artefacts · (c) guardrail demonstrated with a **before/after trace pair** · (d) the ten-question checklist answered in writing · (e) 1-page executive summary · (f) optional 5-min video.

**Evaluation:** runs 30% · governance & guardrails 30% · actuarial substance 25% · communication 15%.

## §1 · Your spec (fill this in FIRST — before any code)
> **Agent name:** …  
> **Who it serves:** (named persona + team)  
> **What it does:** (one sentence)  
> **Source(s) of truth:** (tables/documents the tools read)  
> **What it must NEVER do:** (these become system-prompt rules and guardrail tools)  
> **Output artefact:** (memo? report section? JSON?)  
> **Human gate:** (who reviews, before what action)

In [ ]:
%pip install -q -U agno google-genai xgboost shap statsmodels scikit-learn

In [ ]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("Ready. Team name:", "…")

## §2 · Datasets
Generate any of the three hypothetical ABC books below, or bring a **public** dataset (never confidential data — reread the confidentiality slide).

In [ ]:
# --- ABC Motor 2024: synthetic hypothetical dataset (self-contained, no download needed) ---
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
N = 50_000

motor = pd.DataFrame({
    "policy_id": [f"ABC-MOT-{i:06d}" for i in range(1, N + 1)],
    "vehicle_age_years": rng.integers(0, 16, N),
    "vehicle_make": rng.choice(["Maruti", "Hyundai", "Tata", "Mahindra", "Honda"], N,
                               p=[0.35, 0.25, 0.18, 0.12, 0.10]),
    "vehicle_segment": rng.choice(["Hatchback", "Sedan", "SUV", "MUV"], N,
                                  p=[0.45, 0.25, 0.22, 0.08]),
    "cubic_capacity": rng.choice([998, 1197, 1497, 1997, 2179], N),
    "ncb_pct": rng.choice([0, 20, 25, 35, 45, 50], N, p=[0.30, 0.15, 0.12, 0.15, 0.10, 0.18]),
    "policyholder_age": rng.integers(19, 75, N),
    "policyholder_gender": rng.choice(["M", "F"], N, p=[0.72, 0.28]),
    "region": rng.choice(["Tier1", "Tier2", "Tier3"], N, p=[0.40, 0.35, 0.25]),
    "prior_claims_3y": rng.choice([0, 1, 2, 3], N, p=[0.70, 0.20, 0.07, 0.03]),
})
motor["idv_inr"] = (900_000 * 0.9 ** motor["vehicle_age_years"]
                    * rng.uniform(0.8, 1.2, N)).round(-3)
# earned exposure over each policy's own observation year (mid-term entries/exits)
motor["exposure_years"] = rng.uniform(0.25, 1.0, N).round(3)
# underwriting cohort month — used for the out-of-time split (each cohort observed over its full policy year)
motor["inception_month"] = rng.integers(1, 13, N)

# True frequency model (the "world"): base 8% with realistic loadings
lin = (np.log(0.062)
       + 0.045 * motor["vehicle_age_years"]
       - 0.009 * motor["ncb_pct"]
       + 0.20 * motor["prior_claims_3y"]
       + np.where(motor["region"] == "Tier1", 0.12, np.where(motor["region"] == "Tier3", -0.10, 0.0))
       + np.where(motor["vehicle_segment"] == "SUV", 0.10, 0.0))
motor["claim_count"] = rng.poisson(np.exp(lin) * motor["exposure_years"])
# severity: Gamma, mean ~38k, only where claims exist
sev = rng.gamma(shape=2.0, scale=19_000, size=N)
motor["claim_amount_inr"] = (motor["claim_count"] * sev).round(0)

print("Shape:", motor.shape)
freq = motor.claim_count.sum() / motor.exposure_years.sum()
sev_mean = motor.loc[motor.claim_count > 0, "claim_amount_inr"].sum() / max(motor.claim_count.sum(), 1)
print(f"Portfolio frequency: {freq:.3f} per policy-year | mean severity: INR {sev_mean:,.0f}")
motor.head()

In [ ]:
# ABC Health 2024 — member grain (condensed generator)
rng2 = np.random.default_rng(7)
M = 7678
health = pd.DataFrame({
    "member_id": [f"ABC-HLT-{i:06d}" for i in range(1, M + 1)],
    "member_age": rng2.integers(0, 80, M),
    "relationship": rng2.choice(["Self", "Spouse", "Child", "Parent"], M, p=[0.42, 0.25, 0.23, 0.10]),
    "product_tier": rng2.choice(["Silver", "Gold", "Platinum"], M, p=[0.5, 0.35, 0.15]),
    "pre_existing_flag": rng2.choice([0, 1], M, p=[0.82, 0.18]),
    "region": rng2.choice(["Tier1", "Tier2", "Tier3"], M, p=[0.45, 0.35, 0.20]),
    "exposure_years": rng2.uniform(0.05, 1.0, M).round(3),
})
lin_h = (np.log(0.06) + 0.012 * health.member_age / 10 + 0.46 * health.pre_existing_flag)
health["hospitalisation_count"] = rng2.poisson(np.exp(lin_h) * health.exposure_years)
print("ABC Health 2024:", health.shape)

# ABC Life — term cohort (condensed generator)
L = 20000
life = pd.DataFrame({
    "policy_id": [f"ABC-LIF-{i:06d}" for i in range(1, L + 1)],
    "issue_age": rng2.integers(25, 56, L),
    "smoker_status": rng2.choice(["NS", "S"], L, p=[0.85, 0.15]),
    "premium_frequency": rng2.choice(["Annual", "Monthly"], L, p=[0.55, 0.45]),
    "annualised_premium_inr": rng2.integers(8000, 60000, L),
    "region": rng2.choice(["Tier1", "Tier2", "Tier3"], L),
})
p_lapse = 0.06 + 0.05 * (life.premium_frequency == "Monthly") - 0.0006 * (life.issue_age - 40)
life["lapse_flag"] = rng2.binomial(1, p_lapse.clip(0.01, 0.9))
print("ABC Life cohort:", life.shape)

## §3 · Reusable scaffolds (copy, then adapt)
The guardrail pattern and the call logger — every track needs both.

In [ ]:
# --- Guardrail template: gate a powerful tool behind a deterministic check ---
def make_existence_guardrail(loader, collection_key="factors"):
    """Returns a check function bound to YOUR source of truth (fixes the notebook-04 teaching bug)."""
    def check_in_source(name: str) -> dict:
        """MUST be called before explaining/using any item. Returns existence + valid list."""
        source = loader()
        return {"exists": name in source[collection_key],
                "valid": list(source[collection_key])}
    return check_in_source

# --- Run logger: checklist question 10 ---
import datetime, json, pathlib

def log_run(agent_name, user_query, trace, output, path="agent_run_log.jsonl"):
    rec = {"ts_utc": datetime.datetime.utcnow().isoformat(), "agent": agent_name,
           "query": user_query, "trace": trace, "output": output}
    with open(path, "a") as f:
        f.write(json.dumps(rec) + "\n")
    return rec["ts_utc"]

print("Scaffolds loaded.")

## §4 · The ten-question checklist (submit answered, with evidence)
**Model (1–7)**
1. Purpose — what decision does this support, whose action does it trigger?
2. Data lineage — where did every column come from?
3. Train/test integrity — time-respecting split?
4. Fairness audit — protected AND proxy variables?
5. Interpretability — any single prediction in two sentences?
6. Monitoring — how are drift, decay, bias detected post-launch?
7. Sign-off — named, qualified owner?

**Agent (8–10)**
8. Tool scope — least data, least privilege, for every tool?
9. Guardrails — what mechanically stops hallucination, runaway loops, injection? Where is the human gate?
10. Trace — can you replay every tool call and every number's origin from the log?

## §5 · Submission checklist
- [ ] Runs end-to-end from a clean environment (`Runtime → Disconnect and delete runtime` → `Run all`)
- [ ] Spec (§1) filled and consistent with the system prompt
- [ ] Before/after guardrail trace pair included
- [ ] Ten questions answered with evidence
- [ ] 1-page executive summary (chief-actuary register)
- [ ] All data hypothetical or public; no confidential material anywhere, including prompts

*Office hours: Tue & Thu 6 PM IST on the hub. Bring your trace, not your slides.*